# ADIS Outbreaks Basic EDA

Self-contained first-pass EDA for `data/structured/adis/adis-outbreaks-20260519.csv`. This notebook follows the same structure as the WAHIS and EMPRES-i EDA notebooks while keeping ADIS-specific outbreak burden, diagnostic, status, and administrative fields.

## Setup

Shared plotting, display, and helper functions used throughout the notebook.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

pd.options.display.max_columns = 120
pd.options.display.max_rows = 200
pd.options.mode.chained_assignment = None

NA_VALUES = ['NaN', 'nan', '', ';']

def find_data_path(*candidates: str) -> Path:
    paths = [Path(candidate) for candidate in candidates]
    path = next((candidate for candidate in paths if candidate.exists()), None)
    if path is None:
        tried = '\n'.join(str(candidate) for candidate in paths)
        raise FileNotFoundError(f'Could not find source CSV. Tried:\n{tried}')
    return path


def schema_summary(data: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.DataFrame({
            'dtype': data.dtypes.astype(str),
            'missing': data.isna().sum(),
            'missing_pct': data.isna().mean().mul(100).round(1),
            'n_unique': data.nunique(dropna=True),
        })
        .sort_values(['missing_pct', 'n_unique'], ascending=[False, False])
    )


def top_counts(data: pd.DataFrame, column: str, n: int = 15) -> pd.DataFrame:
    return (
        data[column]
        .fillna('Missing')
        .value_counts(dropna=False)
        .head(n)
        .rename_axis(column)
        .reset_index(name='records')
    )


def plot_top_counts(data: pd.DataFrame, specs: list[tuple[str, int, str]], cols: int = 2) -> None:
    rows = -(-len(specs) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 4.8 * rows), squeeze=False)
    for axis, (column, n, title) in zip(axes.ravel(), specs):
        counts = top_counts(data, column, n=n)
        sns.barplot(data=counts, y=column, x='records', ax=axis, color='#4C78A8')
        axis.set_title(title)
        axis.set_xlabel('Records')
        axis.set_ylabel('')
    for axis in axes.ravel()[len(specs):]:
        axis.set_visible(False)
    plt.tight_layout()


def date_coverage(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame({
        'date_column': columns,
        'min_date': [data[column].min() for column in columns],
        'max_date': [data[column].max() for column in columns],
        'missing': [data[column].isna().sum() for column in columns],
        'missing_pct': [round(data[column].isna().mean() * 100, 1) for column in columns],
    })


def monthly_counts(data: pd.DataFrame, date_column: str, label: str = 'records') -> pd.DataFrame:
    return (
        data.dropna(subset=[date_column])
        .assign(month=lambda frame: frame[date_column].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .rename(label)
        .reset_index()
    )


def monthly_counts_by_category(
    data: pd.DataFrame,
    date_column: str,
    category_column: str,
    top_n: int = 6,
    label: str = 'records',
) -> pd.DataFrame:
    top_values = top_counts(data, category_column, n=top_n)[category_column]
    return (
        data[data[category_column].isin(top_values)]
        .dropna(subset=[date_column])
        .assign(month=lambda frame: frame[date_column].dt.to_period('M').dt.to_timestamp())
        .groupby(['month', category_column])
        .size()
        .rename(label)
        .reset_index()
    )


def country_disease_matrix(
    data: pd.DataFrame,
    country_column: str,
    disease_column: str,
    country_n: int = 12,
    disease_n: int = 8,
) -> pd.DataFrame:
    top_countries = top_counts(data, country_column, n=country_n)[country_column]
    top_diseases = top_counts(data, disease_column, n=disease_n)[disease_column]
    filtered = data[data[country_column].isin(top_countries) & data[disease_column].isin(top_diseases)]
    return pd.crosstab(filtered[country_column], filtered[disease_column])

## Load Data

ADIS uses semicolon-delimited CSV exports, decimal commas for numeric fields, and ISO-style dates.

In [ ]:
DATA_PATH = find_data_path(
    '../data/structured/adis/adis-outbreaks-20260519.csv',
    'data/structured/adis/adis-outbreaks-20260519.csv',
)

raw_df = pd.read_csv(
    DATA_PATH,
    sep=';',
    decimal=',',
    na_values=NA_VALUES,
    keep_default_na=True,
    low_memory=False,
)

print(f'Loaded {len(raw_df):,} rows and {raw_df.shape[1]:,} columns from {DATA_PATH}')
raw_df.head()

## Clean Working Copy

Parse dates, numeric burden fields, boolean-like flags, and derived delay/burden metrics.

In [ ]:
df = raw_df.copy()
df.columns = df.columns.str.strip()

DATE_COLUMNS = [
    'Submitted on',
    'Modified on',
    'Suspicion/Start date',
    'Confirmation date',
    'End date',
    'Result date 1',
]
for column in DATE_COLUMNS:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors='coerce')

NUMERIC_COLUMNS = [
    'Latitude',
    'Longitude',
    'Susceptible 1',
    'Cases 1',
    'Dead 1',
    'Killed 1',
    'Slaughtered 1',
    'Vaccinated 1',
    'Outbreak year',
]
for column in NUMERIC_COLUMNS:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

BOOLEAN_LIKE_COLUMNS = [
    'Outbreak occurring inside an already restricted zone',
    'Approximate location',
    'Suspicion',
    'Clinical signs',
    'Diagnostic tests',
    'Necropsy',
]
for column in BOOLEAN_LIKE_COLUMNS:
    if column in df.columns:
        df[column] = df[column].map({'true': True, 'false': False})

df['confirmation_delay_days'] = (df['Confirmation date'] - df['Suspicion/Start date']).dt.days
df['submission_delay_days'] = (df['Submitted on'] - df['Confirmation date']).dt.days
df['outbreak_duration_days'] = (df['End date'] - df['Suspicion/Start date']).dt.days
df['total_affected_or_removed'] = df[['Cases 1', 'Dead 1', 'Killed 1', 'Slaughtered 1']].sum(axis=1, min_count=1)

duplicate_summary = pd.Series({
    'duplicate_reference': df['Reference'].duplicated().sum(),
    'duplicate_national_reference': df['National reference'].duplicated().sum(),
})

display(df.dtypes.to_frame('dtype'))
display(duplicate_summary.to_frame('count'))

## Dataset Overview And Completeness

In [ ]:
overview = pd.DataFrame({
    'rows': [len(df)],
    'columns': [df.shape[1]],
    'countries': [df['Country/Territory'].nunique()],
    'diseases': [df['Disease name'].nunique()],
    'species': [df['Species 1'].nunique()],
    'continuing_statuses': [df['Status Continuing/Resolved'].nunique()],
    'first_suspicion_date': [df['Suspicion/Start date'].min()],
    'last_suspicion_date': [df['Suspicion/Start date'].max()],
})

display(overview)
display(schema_summary(df).head(35))
display(date_coverage(df, [column for column in DATE_COLUMNS if column in df.columns]))

## Top Categories

In [ ]:
CATEGORY_COLUMNS = [
    'Country/Territory',
    'Disease name',
    'Disease type',
    'Species 1',
    'Status Continuing/Resolved',
    'Epidemiological unit',
]

for column in CATEGORY_COLUMNS:
    display(top_counts(df, column, n=20))

In [ ]:
plot_top_counts(
    df,
    [
        ('Country/Territory', 15, 'Top Countries/Territories'),
        ('Disease name', 12, 'Top Diseases'),
        ('Species 1', 12, 'Top Species'),
        ('Status Continuing/Resolved', 8, 'Outbreak Status'),
    ],
)

## Time Coverage And Trends

In [ ]:
events_by_month = monthly_counts(df, 'Suspicion/Start date', label='outbreak_records')

sns.lineplot(data=events_by_month, x='month', y='outbreak_records', marker='o')
plt.title('ADIS Records By Suspicion/Start Month')
plt.xlabel('Suspicion/start month')
plt.ylabel('Outbreak records')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
events_by_month_and_disease = monthly_counts_by_category(
    df,
    date_column='Suspicion/Start date',
    category_column='Disease name',
    top_n=6,
    label='outbreak_records',
)

sns.lineplot(
    data=events_by_month_and_disease,
    x='month',
    y='outbreak_records',
    hue='Disease name',
    marker='o',
)
plt.title('Top ADIS Diseases Over Time')
plt.xlabel('Suspicion/start month')
plt.ylabel('Outbreak records')
plt.xticks(rotation=45)
plt.legend(title='Disease', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

## Delay Analysis

In [ ]:
delay_columns = ['confirmation_delay_days', 'submission_delay_days', 'outbreak_duration_days']

display(df[delay_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T)

fig, axes = plt.subplots(1, len(delay_columns), figsize=(5.4 * len(delay_columns), 4.2))
for axis, column in zip(axes, delay_columns):
    values = df[column].dropna()
    if not values.empty:
        values = values[(values >= values.quantile(0.01)) & (values <= values.quantile(0.99))]
    sns.histplot(values, bins=30, ax=axis)
    axis.set_title(column.replace('_', ' ').title())
    axis.set_xlabel('Days')
plt.tight_layout()

In [ ]:
df.loc[
    df[delay_columns].notna().any(axis=1),
    ['Country/Territory', 'Disease name', 'Suspicion/Start date', 'Confirmation date', 'Submitted on', *delay_columns],
].sort_values('confirmation_delay_days', ascending=False).head(20)

## Country-Disease Matrix

In [ ]:
matrix = country_disease_matrix(df, 'Country/Territory', 'Disease name')

sns.heatmap(matrix, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Top ADIS Countries/Territories And Diseases')
plt.xlabel('Disease')
plt.ylabel('Country/Territory')
plt.tight_layout()

## Geographic Snapshot

In [ ]:
geo_df = df.dropna(subset=['Longitude', 'Latitude']).copy()

display(geo_df[['Longitude', 'Latitude']].describe())

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=geo_df,
    x='Longitude',
    y='Latitude',
    hue='Disease name',
    hue_order=top_counts(df, 'Disease name', n=8)['Disease name'],
    alpha=0.7,
    s=25,
    linewidth=0,
)
plt.title('ADIS Outbreak Coordinates By Disease')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Disease', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

## Source-Specific Signals

ADIS includes outbreak burden, continuing/resolved status, diagnostic evidence, and administrative divisions.

In [ ]:
burden_columns = ['Susceptible 1', 'Cases 1', 'Dead 1', 'Killed 1', 'Slaughtered 1', 'Vaccinated 1', 'total_affected_or_removed']

display(df[burden_columns].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T)

source_specific = {
    'continuing_by_country': pd.crosstab(df['Country/Territory'], df['Status Continuing/Resolved'], dropna=False).sort_index(),
    'diagnostics_by_disease': pd.crosstab(df['Disease name'], df['Diagnostic tests'], dropna=False),
    'top_admin1': top_counts(df, 'Administrative division level 1', 20),
    'restricted_zone_by_disease': pd.crosstab(
        df['Disease name'],
        df['Outbreak occurring inside an already restricted zone'],
        dropna=False,
    ),
}

display(source_specific['continuing_by_country'].tail(20))
display(source_specific['diagnostics_by_disease'].head(20))
display(source_specific['top_admin1'])
display(source_specific['restricted_zone_by_disease'].head(20))

## Quick Filters

Set any filter to a string value, or leave it as `None` to ignore that filter.

In [ ]:
country_filter = None
disease_filter = None
status_filter = None
species_filter = None

filtered = df.copy()
if country_filter:
    filtered = filtered[filtered['Country/Territory'].eq(country_filter)]
if disease_filter:
    filtered = filtered[filtered['Disease name'].eq(disease_filter)]
if status_filter:
    filtered = filtered[filtered['Status Continuing/Resolved'].eq(status_filter)]
if species_filter:
    filtered = filtered[filtered['Species 1'].eq(species_filter)]

filtered.sort_values('Submitted on', ascending=False).head(50)